# 02 — Model Training

## Purpose

This notebook trains baseline resume-screening models on the synthetic dataset generated in `01_data_generation.ipynb`.

The primary baseline is Logistic Regression using resume-derived features only. Protected attributes such as age are excluded from model training by design.

This notebook focuses on overall predictive performance. Group-level fairness metrics are evaluated separately in `03_fairness_evaluation.ipynb`.

## Scope
- Load the persisted dataset
- Define feature sets and target variable
- Build preprocessing pipelines for mixed numeric, categorical, and boolean features
- Train a baseline Logistic Regression model
- Save fitted model and key outputs for downstream fairness evaluation

## Important Constraint

Protected attributes (`true_age`, `age_group`) are explicitly excluded from model features. These variables are retained only for post-hoc fairness evaluation.

## Outputs

- Trained baseline model artifact
- Test-set predictions
- Overall classification metrics

These artifacts will be used in the fairness evaluation and ablation notebooks.

## Imports and Configuration

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import json
import joblib
import pandas as pd
import numpy as np
import pyarrow

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

## Load Dataset

In [3]:
data_path = PROJECT_ROOT / "data" / "baseline" / "synthetic_resumes_full.parquet"
df = pd.read_parquet(data_path)

print(df.shape)
df.head()

(10000, 61)


,candidate_id,application_year,target_role_family,target_role_level,region,age_group,true_age,highest_degree,school_tier,graduation_year,...,salary_expectation_usd,willing_to_relocate,remote_only,estimated_start_year,tech_recency_score,leadership_signal_score,stability_score,callback,interview,offer
0,cand_000000,2025,SWE,Senior,US-West,30-39,31,None,4,2016,...,121909,False,False,2019,0.507277,0.000000,0.318828,False,<NA>,<NA>
1,cand_000001,2025,IT,Mid,UK,<30,22,AA,1,2024,...,50000,True,False,2025,0.708427,0.000000,0.439782,True,<NA>,<NA>
2,cand_000002,2025,Security,Staff,US-East,30-39,35,PhD,2,2011,...,145311,False,False,2014,0.631192,0.028523,0.310756,True,<NA>,<NA>
3,cand_000003,2025,PM,Staff,UK,50-59,53,AA,1,1993,...,242766,False,False,2006,0.514025,0.215608,0.243552,False,<NA>,<NA>
4,cand_000004,2025,PM,Mid,EU,30-39,33,HS,3,2013,...,93707,True,True,2020,0.407198,0.000000,-0.055558,False,<NA>,<NA>


In [4]:
required_cols = ["callback", "age_group"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

## Define Feature Sets and Target

In [5]:
FEATURES_WITH_GRAD_YEAR = [
    "application_year",
    "target_role_family",
    "target_role_level",
    "region",
    "highest_degree",
    "graduation_year",
    "school_tier",
    "gpa_bucket",
    "years_experience_total",
    "years_experience_relevant",
    "num_employers",
    "avg_tenure_years",
    "months_since_last_role",
    "num_gaps_over_6mo",
    "most_recent_title",
    "most_recent_company_size",
    "management_years",
    "reports_max",
    "num_skills_listed",
    "num_programming_languages",
    "num_cloud_platforms",
    "num_databases",
    "skill_python",
    "skill_java",
    "skill_javascript",
    "skill_go",
    "skill_kubernetes",
    "skill_aws",
    "skill_gcp",
    "skill_azure",
    "skill_sql",
    "skill_spark",
    "skill_terraform",
    "skill_linux",
    "skill_ml",
    "legacy_tech_count",
    "modern_tech_count",
    "cert_count",
    "has_top_cloud_cert",
    "github_url_present",
    "portfolio_url_present",
    "open_source_mentions",
    "patent_count",
    "resume_word_count",
    "bullet_count",
    "quantified_impact_count",
    "keyword_match_score",
    "format_clean_score",
    "salary_expectation_usd",
    "willing_to_relocate",
    "remote_only",
]

In [6]:
TARGET = "callback"

X = df[FEATURES_WITH_GRAD_YEAR].copy()
y = df[TARGET].astype(int)

print("X shape:", X.shape)
print("y mean:", y.mean())

X shape: (10000, 51)
y mean: 0.5


## Train/Test Split

In [7]:
X_train, X_test, y_train, y_test, age_train, age_test = train_test_split(
    X,
    y,
    df["age_group"],
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(8000, 51) (2000, 51)
0.5 0.5


## Preprocessing Pipeline (Encoding + Scaling)

In [8]:
numeric_features = X.select_dtypes(include=["int8", "int16", "int32", "float32", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["category", "object", "string"]).columns.tolist()
boolean_features = X.select_dtypes(include=["bool", "boolean"]).columns.tolist()

print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Boolean:", len(boolean_features))

Numeric: 25
Categorical: 7
Boolean: 19


In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

boolean_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bool", boolean_transformer, boolean_features),
    ]
)

## Train Baseline Models

In [10]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ]
)

logreg_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['application_year',
                                                   'graduation_year',
                                                   'school_tier',
                                                   'years_experience_total',
                                                   'years_experience_relevant',
                                                   'num_employers',
                                                   'avg_tenure_years',
                                                   'months_since_last_role',
                                                   'num_gaps_over_6mo',
                                                   'ma...
                                                   'skill_javascript',
                                                   'skill_go',
                                                   'skill_kubernetes',
                                                   'skill_aws', 'skill_gcp',
                                                   'skill_azure', 'skill_sql',
                                                   'skill_spark',
                                                   'skill_terraform',
                                                   'skill_linux', 'skill_ml',
                                                   'has_top_cloud_cert',
                                                   'github_url_present',
                                                   'portfolio_url_present',
                                                   'open_source_mentions',
                                                   'willing_to_relocate',
                                                   'remote_only'])])),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])

## Performance Evaluation (Overall)

In [11]:
y_pred = logreg_pipeline.predict(X_test)
y_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
}

metrics_df = pd.DataFrame([metrics])
metrics_df

,accuracy,precision,recall,f1,roc_auc
0,0.872,0.869781,0.875,0.872383,0.948213


In [12]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[869 131]
 [125 875]]


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1000
           1       0.87      0.88      0.87      1000

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000



This notebook focuses on overall predictive performance only. Group-level fairness metrics will be computed separately in `03_fairness_evaluation.ipynb`.

## Save Models and Metrics (Optional)

In [14]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "logreg_with_grad_year.joblib"
joblib.dump(logreg_pipeline, model_path)

print(f"Saved model to: {model_path}")

Saved model to: /home/marshall/Projects/ucbaicert/ucbai-cs-resumefilter/models/logreg_with_grad_year.joblib


In [15]:
predictions_df = X_test.copy()
predictions_df["y_true"] = y_test.values
predictions_df["y_pred"] = y_pred
predictions_df["y_prob"] = y_prob
predictions_df["age_group"] = age_test.values

predictions_path = PROJECT_ROOT / "data" / "baseline" / "logreg_with_grad_year_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print(f"Saved predictions to: {predictions_path}")

Saved predictions to: /home/marshall/Projects/ucbaicert/ucbai-cs-resumefilter/data/baseline/logreg_with_grad_year_predictions.parquet


In [16]:
reports_tables_dir = PROJECT_ROOT / "reports" / "tables"
reports_tables_dir.mkdir(parents=True, exist_ok=True)

metrics_path = reports_tables_dir / "logreg_with_grad_year_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"Saved metrics to: {metrics_path}")

Saved metrics to: /home/marshall/Projects/ucbaicert/ucbai-cs-resumefilter/reports/tables/logreg_with_grad_year_metrics.csv
